# YipitData Take-Home Assessment: End-to-End Pipeline & Hybrid Search

This notebook walks through the entire end-to-end flow of the project:
1. **Stage 1: Core ETL Pipeline**: Ingestion, cleaning, metadata enrichment, schema validation, and exporting the enriched dataset.
2. **Stage 2: Hybrid Semantic Search**: Generates sentence embeddings, computes top 3 similar articles, loads data into DuckDB, applies SQL filters, and exports the final canonical dataset.
3. **Stage 3: Interactive Search Demos**: Demonstrates how to use the `SemanticSearchEngine` programmatically for semantic and hybrid searches.

In [1]:
import sys
from pathlib import Path

# Setup paths relative to the current workspace root
project_root = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
sys.path.extend([
    str(project_root),
    str(project_root / "search"),
    str(project_root / "pipeline")
])

import pandas as pd
import numpy as np
import duckdb
import pipeline
import search
from search_engine import SemanticSearchEngine

/Users/huangpan/Documents/YipitData/takehome_yd/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1: Run Core ETL Pipeline
We run the core ETL pipeline on the raw tech news CSV to output the enriched dataset.

In [2]:
input_csv = project_root / "pipeline/data/input/tech_news.csv"
metadata_json = project_root / "pipeline/data/input/company_metadata.json"
enriched_csv = project_root / "pipeline/data/output/ai_articles_enriched.csv"

print("Running ETL pipeline...")
pipeline.run(
    input_path=input_csv,
    metadata_path=metadata_json,
    output_path=enriched_csv,
    apply_filter=False
)

# Display a preview of the enriched dataset
df_enriched = pd.read_csv(enriched_csv)
print(f"\nETL Pipeline complete. Loaded preview ({df_enriched.shape[0]} rows, {df_enriched.shape[1]} columns):")
df_enriched.head(3)

18:22:14  INFO      pipeline — === YipitData ETL Pipeline — START ===
18:22:14  INFO      pipeline — [1/4] Ingesting data…
18:22:14  INFO      pipeline — Ingested 500 rows from '/Users/huangpan/Documents/YipitData/takehome_yd/pipeline/data/input/tech_news.csv'
18:22:14  INFO      pipeline — [1/4] Validating raw schema…
18:22:14  INFO      validation.schemas — [raw] All 500 rows passed schema validation ✓
18:22:14  INFO      pipeline — [2/4] Cleaning data…
18:22:14  INFO      pipeline — Cleaning complete — revenue errors: 0, date errors: 0, category errors: 0
18:22:14  INFO      pipeline — [2/4] Validating cleaned schema…
18:22:14  INFO      validation.schemas — [clean] All 500 rows passed schema validation ✓
18:22:14  INFO      pipeline — [3/4] Enriching with metadata…
18:22:14  INFO      enrichment.metadata — Loaded metadata for 21 companies
18:22:14  INFO      enrichment.metadata — Metadata join complete: 500 matched, 0 unmatched
18:22:14  INFO      pipeline — [3/4] Validating enrich

Running ETL pipeline...

ETL Pipeline complete. Loaded preview (500 rows, 22 columns):


,article_id,title,company_name,published_date,pub_year,pub_quarter,pub_month,category,revenue_usd,summary,...,word_count,metadata_matched,meta_founded_year,meta_headquarters,meta_employee_count,meta_industry,meta_is_public,meta_stock_ticker,company_age,company_size_category
0,ART0001,Scale AI Raises Series D Funding Round,Scale AI,2020-02-21,2020,1,2,FinTech,0,The company demonstrated significant advances ...,...,2569.0,True,2003,"Berlin, Germany",23379,AI/ML,False,NaN,17,Medium
1,ART0002,DataRobot Announces Breakthrough in Large Lang...,DataRobot,2023-02-23,2023,1,2,SaaS_Software,312151151,Strategic acquisition strengthens competitive ...,...,1878.0,True,2012,"New York, NY",23471,Cloud Computing,True,NaN,11,Medium
2,ART0003,Meta AI Raises Series D Funding Round,Meta AI,2022-02-17,2022,1,2,Data_Analytics,0,Leadership outlines vision for responsible AI ...,...,1981.0,True,2001,"Seattle, WA",20240,Data Analytics,True,NaN,21,Medium


## Part 2: Run Hybrid Semantic Search Pipeline
Now we run the search pipeline, which will:
1. Read the enriched articles CSV.
2. Generate 384-dimensional sentence embeddings using the `all-MiniLM-L6-v2` model.
3. Compute the top 3 similar articles for each article.
4. Load everything into a DuckDB vector storage.
5. Query & export the filtered canonical schema to `search/output/filtered_ai_articles_with_embeddings.csv`.

In [3]:
search_output_csv = project_root / "search/output/filtered_ai_articles_with_embeddings.csv"

print("Running Hybrid Semantic Search Pipeline...")
search.run_pipeline(
    input_path=enriched_csv,
    output_path=search_output_csv,
    model_name="all-MiniLM-L6-v2"
)

# Preview the exported filtered articles
df_filtered = pd.read_csv(search_output_csv)
print(f"\nSemantic search pipeline complete. Filtered size: {df_filtered.shape[0]} rows.")
df_filtered[["article_id", "title", "company_name", "category", "revenue_usd", "top_similar_articles"]].head(5)

18:23:03  INFO      semantic_pipeline — === Starting Hybrid Semantic Search Pipeline ===
18:23:03  INFO      semantic_pipeline — Loading enriched articles from '/Users/huangpan/Documents/YipitData/takehome_yd/pipeline/data/output/ai_articles_enriched.csv'...
18:23:03  INFO      semantic_pipeline — Loaded 500 rows.
18:23:03  INFO      semantic_pipeline — Generating text embeddings using 'all-MiniLM-L6-v2' model...
18:23:03  INFO      sentence_transformers.SentenceTransformer — Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Running Hybrid Semantic Search Pipeline...


/Users/huangpan/Documents/YipitData/takehome_yd/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
18:23:05  INFO      sentence_transformers.SentenceTransformer — Use pytorch device_name: mps
18:23:10  INFO      semantic_pipeline — Generated embeddings with shape (500, 384).
18:23:10  INFO      semantic_pipeline — Calculating top 3 most similar articles for each row...
18:23:11  INFO      semantic_pipeline — Loading data into DuckDB...
18:23:11  INFO      semantic_pipeline — Applying SQL filters & renaming columns using DuckDB...
18:23:11  INFO      semantic_pipeline — Filtered dataset size: 48 rows.
18:23:11  INFO      semantic_pipeline — Successfully exported 48 rows to '/Users/huangpan/Documents/YipitData/takehome_yd/search/output/filtered_ai_articles_with_emb


Semantic search pipeline complete. Filtered size: 48 rows.


,article_id,title,company_name,category,revenue_usd,top_similar_articles
0,ART0004,Confluent Reports Record Revenue Growth,Confluent,AI_ML,100000000,"[""ART0420"", ""ART0391"", ""ART0096""]"
1,ART0009,MongoDB Expands into European Market,MongoDB,AI_ML,80000000,"[""ART0412"", ""ART0090"", ""ART0304""]"
2,ART0029,NVIDIA Announces Breakthrough in Large Languag...,NVIDIA,Data_Analytics,4406063729,"[""ART0154"", ""ART0183"", ""ART0214""]"
3,ART0033,SpaceX Achieves Profitability Milestone,SpaceX,Cloud_Computing,90000000,"[""ART0060"", ""ART0452"", ""ART0269""]"
4,ART0036,NVIDIA Launches New AI-Powered Product,NVIDIA,AI_ML,1170000000,"[""ART0214"", ""ART0066"", ""ART0411""]"


## Part 3: Programmatic Usage & Interactive Search Demonstration
We initialize a `SemanticSearchEngine` and perform interactive semantic search and hybrid search queries.

In [4]:
# Initialize the engine
engine = SemanticSearchEngine(model_name="all-MiniLM-L6-v2")

# Load the enriched articles
texts = (df_enriched["title"].fillna("") + ". " + df_enriched["summary"].fillna("")).tolist()
embeddings = engine.generate_embeddings(texts)

# Calculate top 3 most similar articles (excluding self) for each article
print("Calculating top 3 most similar articles for each row...")
similarity_matrix = np.dot(embeddings, embeddings.T)
norms = np.linalg.norm(embeddings, axis=1)
norms_matrix = np.outer(norms, norms)
similarity_matrix = similarity_matrix / np.maximum(norms_matrix, 1e-12)

top_similar_list = []
for idx in range(len(df_enriched)):
    sim_scores = similarity_matrix[idx]
    sorted_indices = np.argsort(sim_scores)[::-1]
    filtered_indices = [i for i in sorted_indices if i != idx]
    top_3_indices = filtered_indices[:3]
    top_3_ids = df_enriched.iloc[top_3_indices]["article_id"].tolist()
    top_similar_list.append(top_3_ids)

df_enriched["top_similar_articles"] = top_similar_list

# Load data into engine
engine.load_data(df_enriched, embeddings)

18:25:50  INFO      sentence_transformers.SentenceTransformer — Load pretrained SentenceTransformer: all-MiniLM-L6-v2
/Users/huangpan/Documents/YipitData/takehome_yd/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
18:25:51  INFO      sentence_transformers.SentenceTransformer — Use pytorch device_name: mps


Calculating top 3 most similar articles for each row...


In [11]:
# Vector similarity search for "NVIDIA financial results"
query = "NVIDIA financial results"
print(f"Executing Vector Similarity Search for: '{query}'")
results = engine.find_similar_articles(query, top_k=3)

# Display results in a readable format
results_df = []
for art_id, score in results:
    row = df_enriched[df_enriched["article_id"] == art_id].iloc[0]
    results_df.append({
        "Article ID": art_id,
        "Score": f"{score:.4f}",
        "Title": row["title"],
        "Company": row["company_name"],
        "Summary": row["summary"]
    })
pd.DataFrame(results_df)

Executing Vector Similarity Search for: 'NVIDIA financial results'


,Article ID,Score,Title,Company,Summary
0,ART0006,0.7045,NVIDIA Achieves Profitability Milestone,NVIDIA,Financial results exceed analyst expectations ...
1,ART0183,0.5691,NVIDIA Expands into European Market,NVIDIA,Company reaches important financial milestone ...
2,ART0297,0.5511,NVIDIA Acquires Competitor for Undisclosed Sum,NVIDIA,Investors show strong confidence in the compan...


### Vector Similarity Search using an Article ID
We can also query using a specific `article_id`. The engine resolves the ID, retrieves its precomputed embedding, and finds the most similar articles in the corpus.

In [15]:
# Query by article_id (e.g., "ART0003" which corresponds to NVIDIA profitability results)
query_id = "ART0001"
print(f"Executing Vector Similarity Search by Article ID: '{query_id}'")

# Find top 4 similar articles (the first result should be the article itself)
id_results = engine.find_similar_articles(query_id, top_k=4)

# Display results
id_results_df = []
for art_id, score in id_results:
    row = df_enriched[df_enriched["article_id"] == art_id].iloc[0]
    id_results_df.append({
        "Article ID": art_id,
        "Score": f"{score:.4f}",
        "Title": row["title"],
        "Company": row["company_name"],
        "Summary": row["summary"]
    })
pd.DataFrame(id_results_df)

Executing Vector Similarity Search by Article ID: 'ART0001'


,Article ID,Score,Title,Company,Summary
0,ART0001,1.0000,Scale AI Raises Series D Funding Round,Scale AI,The company demonstrated significant advances ...
1,ART0072,0.9105,Scale AI Raises Series D Funding Round,Scale AI,Company reaches important financial milestone ...
2,ART0219,0.9105,Scale AI Raises Series D Funding Round,Scale AI,Company reaches important financial milestone ...
3,ART0079,0.8625,Scale AI Raises Series D Funding Round,Scale AI,Investors show strong confidence in the compan...


### Hybrid Search (SQL Filters + Vector Similarity)
We apply standard SQL filters first, then perform similarity ranking on the remaining records.

In [17]:
# Hybrid Search: SQL Filters + Vector Similarity
query = "AI"
sql_filter = "pub_year BETWEEN 2022 AND 2024 AND revenue_usd >= 50000000"

print(f"Executing Hybrid Search:")
print(f"  Query: '{query}'")
print(f"  SQL Filter: {sql_filter}")

hybrid_results = engine.hybrid_search(query, sql_filter, top_k=5)

# Display results
hybrid_df = []
for art_id, score in hybrid_results:
    row = df_enriched[df_enriched["article_id"] == art_id].iloc[0]
    hybrid_df.append({
        "Article ID": art_id,
        "Score": f"{score:.4f}",
        "Title": row["title"],
        "Company": row["company_name"],
        "Revenue (USD)": f"${row['revenue_usd']:,}",
        "Year": row["pub_year"],
        "Summary": row["summary"]
    })
pd.DataFrame(hybrid_df)

Executing Hybrid Search:
  Query: 'AI'
  SQL Filter: pub_year BETWEEN 2022 AND 2024 AND revenue_usd >= 50000000


,Article ID,Score,Title,Company,Revenue (USD),Year,Summary
0,ART0170,0.5791,Anthropic Launches New AI-Powered Product,Anthropic,"$440,000,000",2022,New product features leverage cutting-edge AI ...
1,ART0141,0.5261,Anthropic Expands into European Market,Anthropic,"$8,790,000,000",2022,New product features leverage cutting-edge AI ...
2,ART0322,0.5073,Meta AI Launches New AI-Powered Product,Meta AI,"$80,000,000",2023,Investors show strong confidence in the compan...
3,ART0231,0.5023,Meta AI Reports Record Revenue Growth,Meta AI,"$5,092,056,379",2022,New product features leverage cutting-edge AI ...
4,ART0031,0.5005,Meta AI CEO Discusses Future of AI,Meta AI,"$805,083,997",2023,Leadership outlines vision for responsible AI ...


### Direct Pipeline SQL Query Execution
We execute the exact pipeline SQL query (`test_pipeline_sql_query`) directly using DuckDB to retrieve the canonical schema.

In [18]:
# Pipeline SQL query matching test_pipeline_sql_query
pipeline_query = """
SELECT 
    article_id,
    title,
    company_name,
    published_date,
    category,
    revenue_usd,
    summary,
    url,
    meta_industry AS industry,
    meta_founded_year AS founded_year,
    meta_headquarters AS headquarters,
    meta_employee_count AS employee_count,
    meta_is_public AS is_public,
    meta_stock_ticker AS stock_ticker,
    company_age,
    company_size_category,
    embedding,
    top_similar_articles
FROM articles
WHERE 
    (category = 'AI_ML' OR meta_industry = 'AI/ML')
    AND pub_year BETWEEN 2022 AND 2024
    AND revenue_usd >= 50000000
"""

print("Executing SQL query on DuckDB articles table...")
sql_df = engine.execute_sql(pipeline_query)

print(f"\nQuery executed successfully. Returned {sql_df.shape[0]} matching articles.")
sql_df[["article_id", "title", "company_name", "industry", "revenue_usd", "is_public"]].head(5)

Executing SQL query on DuckDB articles table...

Query executed successfully. Returned 48 matching articles.


,article_id,title,company_name,industry,revenue_usd,is_public
0,ART0004,Confluent Reports Record Revenue Growth,Confluent,Cloud Computing,100000000,False
1,ART0009,MongoDB Expands into European Market,MongoDB,SaaS,80000000,False
2,ART0029,NVIDIA Announces Breakthrough in Large Languag...,NVIDIA,AI/ML,4406063729,False
3,ART0033,SpaceX Achieves Profitability Milestone,SpaceX,AI/ML,90000000,False
4,ART0036,NVIDIA Launches New AI-Powered Product,NVIDIA,AI/ML,1170000000,False
